# 06 – Final Evaluation on the Test Set
In this notebook, we evaluate our final model once on the test set, which we have not used until now. The final model is the one chosen with cross-validation in notebooks 03 to 05: Gradient Boosting with all 30 features. Notebook 04 showed that feature selection does not improve the score, and notebook 05 showed that ensembles and the neural network are not better. We compare it with the logistic regression baseline, give confidence intervals, and test how the model reacts when the data changes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (precision_recall_curve, recall_score, precision_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
X_train, y_train = train.drop(columns="class"), train["class"]
X_test, y_test = test.drop(columns="class"), test["class"]
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 1. Final models and decision thresholds
We use the hyperparameters found in notebook 03. We choose the thresholds again on the training set only, with the same method as before: the best precision among the thresholds that reach 95% recall.

In [ ]:
final_models = {
    "Logistic regression (baseline)": LogisticRegression(C=10, max_iter=1000, random_state=SEED),
    "Gradient Boosting (final)": HistGradientBoostingClassifier(
        learning_rate=0.1, max_iter=200, max_depth=6, l2_regularization=0.0, random_state=SEED),
}

cv_thr = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

def pick_threshold(model, target_recall=0.95):
    proba = cross_val_predict(model, X_train, y_train, cv=cv_thr, method="predict_proba")[:, 1]
    prec, rec, thr = precision_recall_curve(y_train, proba)
    ok = rec[:-1] >= target_recall
    return thr[np.argmax(np.where(ok, prec[:-1], -1))]

thresholds = {name: pick_threshold(m) for name, m in final_models.items()}
for name, t in thresholds.items():
    print(f"{name:32s} threshold = {t:.3f}")

## 2. Training on the whole training set, then one evaluation on the test set

In [ ]:
def scores(y_true, proba, t):
    y_hat = (proba >= t).astype(int)
    return {"recall": recall_score(y_true, y_hat), "precision": precision_score(y_true, y_hat),
            "f1": f1_score(y_true, y_hat), "roc_auc": roc_auc_score(y_true, proba)}

test_proba, test_scores = {}, {}
for name, model in final_models.items():
    model.fit(X_train, y_train)
    test_proba[name] = model.predict_proba(X_test)[:, 1]
    test_scores[name] = scores(y_test, test_proba[name], thresholds[name])

pd.DataFrame(test_scores).T.round(3)

## 3. Confidence intervals (bootstrap)
One score alone does not show how certain it is. We draw 1,000 random samples of the test set and compute the metrics each time, which gives a 95% confidence interval. We also compute the difference between the two models, to check if Gradient Boosting is really better.

In [ ]:
rng = np.random.RandomState(SEED)
names = list(final_models)
boot = {n: [] for n in names}
diff = []
y_arr = y_test.to_numpy()
for _ in range(1000):
    idx = rng.randint(0, len(y_arr), len(y_arr))
    s = {n: scores(y_arr[idx], test_proba[n][idx], thresholds[n]) for n in names}
    for n in names:
        boot[n].append(s[n])
    diff.append({m: s[names[1]][m] - s[names[0]][m] for m in s[names[0]]})

rows = {}
for n in names:
    b = pd.DataFrame(boot[n])
    rows[n] = {m: f"{test_scores[n][m]:.3f} [{b[m].quantile(.025):.3f}, {b[m].quantile(.975):.3f}]"
               for m in b.columns}
d = pd.DataFrame(diff)
rows["Difference (GB − baseline)"] = {m: f"{d[m].mean():+.3f} [{d[m].quantile(.025):+.3f}, {d[m].quantile(.975):+.3f}]"
                                       for m in d.columns}
ci_table = pd.DataFrame(rows).T
ci_table

If the confidence interval of the difference does not contain 0, the improvement of Gradient Boosting over the baseline is not due to chance on this test set.

## 4. Confusion matrices on the test set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, n in zip(axes, names):
    y_hat = (test_proba[n] >= thresholds[n]).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_hat),
                           display_labels=["legitimate", "phishing"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.savefig(FIG_DIR / "test_confusion_matrices.png", dpi=150); plt.show()

for n in names:
    tn, fp, fn, tp = confusion_matrix(y_test, (test_proba[n] >= thresholds[n]).astype(int)).ravel()
    print(f"{n:32s} missed phishing: {fn:3d} | false alarms: {fp:3d}")

## 5. Comparison with cross-validation
If the test scores are close to the cross-validation scores of notebook 03, the model generalises well and our protocol did not overestimate it.

In [ ]:
cv_ref = {"Logistic regression (baseline)": {"f1": 0.923, "roc_auc": 0.976},
          "Gradient Boosting (final)": {"f1": 0.951, "roc_auc": 0.993}}   # notebook 03, threshold 0.5
pd.DataFrame({n: {"CV ROC-AUC (nb 03)": cv_ref[n]["roc_auc"], "Test ROC-AUC": round(test_scores[n]["roc_auc"], 3)}
              for n in names}).T

## 6. Stress test: what if phishing websites start using HTTPS?
Our data was collected around 2012–2015. Today, most phishing websites use HTTPS. HTTPS is one of the two most important features of our model, so we simulate this change: on the test set, we set HTTPS to its "legitimate" value (1) for every phishing website, and we measure the recall again.

In [ ]:
X_shift = X_test.copy()
X_shift.loc[y_test == 1, "HTTPS"] = 1

shift_rows = {}
for n, model in final_models.items():
    p = model.predict_proba(X_shift)[:, 1]
    shift_rows[n] = {"recall (original test)": round(test_scores[n]["recall"], 3),
                     "recall (all phishing use HTTPS)": round(recall_score(y_test, (p >= thresholds[n]).astype(int)), 3)}
pd.DataFrame(shift_rows).T

This shows a real failure mode: if attackers copy the signs of legitimate websites on the most important features, the recall drops. A model trained on old data must be retrained regularly on recent phishing websites.

## Conclusions
- On the test set, Gradient Boosting reaches a recall of 0.947, a precision of 0.969, an F1-score of 0.958 and a ROC-AUC of 0.994. These scores are very close to the cross-validation scores, so the model generalises well.
- Compared with the baseline, Gradient Boosting has the same recall (the difference is not significant) but a much better precision (+0.10): 18 false alarms instead of 84 on the test set.
- The recall (0.947) is just below our 95% target, but the 95% confidence interval [0.927, 0.964] contains 0.95: with about 600 phishing websites in the test set, we cannot say that the target is missed or reached.
- The stress test shows the main limit: if all phishing websites use HTTPS, the recall drops to about 0.82. The model depends on features that attackers can copy, and it must be retrained on recent data.

In [ ]:
gb, lr = test_scores[names[1]], test_scores[names[0]]
print(f"Final model (Gradient Boosting) on the test set: recall = {gb['recall']:.3f}, "
      f"precision = {gb['precision']:.3f}, F1 = {gb['f1']:.3f}, ROC-AUC = {gb['roc_auc']:.3f}")
print(f"Baseline (logistic regression):                  recall = {lr['recall']:.3f}, "
      f"precision = {lr['precision']:.3f}, F1 = {lr['f1']:.3f}, ROC-AUC = {lr['roc_auc']:.3f}")